In [7]:
# Importing necessary libraries

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import boto3
from sagemaker import get_execution_role
import seaborn as sns

### S3 connection

In [3]:
# Developing the connection with S3 where we had stored our dataset.

conn = boto3.client('s3')
bucket_name = "mlc-project-flight-data-2024"

In [ ]:
content = conn.list_objects(Bucket=bucket_name)["Contents"] # Listing objects in the S3 bucket

In [ ]:
content # Displaying the content of the S3 bucket

[{'Key': 'flight_data_2024.csv',
  'LastModified': datetime.datetime(2025, 11, 21, 19, 26, 39, tzinfo=tzlocal()),
  'ETag': '"27aae244de0f43411ae0ad7c5d543617-77"',
  'ChecksumAlgorithm': ['CRC64NVME'],
  'ChecksumType': 'FULL_OBJECT',
  'Size': 1309010752,
  'StorageClass': 'STANDARD',
  'Owner': {'ID': '71497476b2ed1deb7d6cf99b12a744a04a5fbbb5001b17f994dd6900b136f1a6'}},
 {'Key': 'flight_data_2024_sample.csv',
  'LastModified': datetime.datetime(2025, 11, 21, 19, 26, 40, tzinfo=tzlocal()),
  'ETag': '"ffa51b3a5327aab85b2e2ff824420ac2"',
  'ChecksumAlgorithm': ['CRC64NVME'],
  'ChecksumType': 'FULL_OBJECT',
  'Size': 1850581,
  'StorageClass': 'STANDARD',
  'Owner': {'ID': '71497476b2ed1deb7d6cf99b12a744a04a5fbbb5001b17f994dd6900b136f1a6'}}]

In [ ]:
for data in content:
    print(data['Key']) # Printing the keys of the objects in the S3 bucket

flight_data_2024.csv
flight_data_2024_sample.csv


### Data preprocessing

In [ ]:
# Define output
input_key   = "flight_data_2024.csv"
input_uri   = f"s3://{bucket_name}/{input_key}"

out_prefix  = "preprocessed"  # where we’ll write outputs
clean_key   = f"{out_prefix}/flight_data_2024_clean.csv"
profile_key = f"{out_prefix}/profiles/column_profile.csv"
eda_key     = f"{out_prefix}/profiles/quick_eda.json"

s3 = boto3.client("s3") # S3 client

In [ ]:
# Booking-time candidates (year & month intentionally NOT included since there is only one year and two months data available)
FEATURE_CANDIDATES = [
    "day_of_month","day_of_week","fl_date",
    "op_unique_carrier","op_carrier_fl_num",
    "origin","origin_city_name","origin_state_nm","dest",
    "dest_city_name","dest_state_nm","crs_dep_time","crs_arr_time",
    "crs_elapsed_time","distance"
]

LABEL_CANDIDATES = ["is_late","arr_delay"] # Target labels

In [ ]:
header = pd.read_csv(input_uri, nrows=0).columns.tolist() # Read only the header row
present_features = [c for c in FEATURE_CANDIDATES if c in header] # Check which feature columns are present
present_labels   = [c for c in LABEL_CANDIDATES if c in header] # Check which label columns are present

if not present_features:
    raise ValueError("None of the expected feature columns are present.") # Check for feature columns

if not present_labels:
    raise ValueError("Need either 'is_late' or 'arr_delay' in the CSV.") # Check for label columns

usecols = present_features + present_labels # Columns to read

df = pd.read_csv(input_uri, usecols=usecols) # Read the data
print(df.shape, "rows,cols")
df.head() # Display the first few rows of the dataframe

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/fsspec/registry.py:298: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


(7079081, 16) rows,cols


,day_of_month,day_of_week,fl_date,op_unique_carrier,op_carrier_fl_num,origin,origin_city_name,origin_state_nm,dest,dest_city_name,dest_state_nm,crs_dep_time,crs_arr_time,arr_delay,crs_elapsed_time,distance
0,1,1,2024-01-01,9E,4814.0,JFK,"New York, NY",New York,DTW,"Detroit, MI",Michigan,1252,1508,-19.0,136.0,509.0
1,1,1,2024-01-01,9E,4815.0,MSP,"Minneapolis, MN",Minnesota,CLE,"Cleveland, OH",Ohio,1015,1325,-30.0,130.0,622.0
2,1,1,2024-01-01,9E,4817.0,JFK,"New York, NY",New York,RIC,"Richmond, VA",Virginia,1415,1601,-20.0,106.0,288.0
3,1,1,2024-01-01,9E,4817.0,RIC,"Richmond, VA",Virginia,JFK,"New York, NY",New York,1650,1841,-42.0,111.0,288.0
4,1,1,2024-01-01,9E,4818.0,DTW,"Detroit, MI",Michigan,MKE,"Milwaukee, WI",Wisconsin,1015,1034,-14.0,79.0,237.0


In [12]:
# Light cleaning & type coercion

# strip whitespace in string-like categorical columns
for c in ["op_unique_carrier","origin","origin_city_name","origin_state_nm","dest","dest_city_name","dest_state_nm"]:
    if c in df.columns:
        df[c] = df[c].astype("string").str.strip().astype("category")

# Ensure time fields are valid HHMM integers; coerce bad values to NA
for tcol in ["crs_dep_time","crs_arr_time"]:
    if tcol in df.columns:
        df[tcol] = pd.to_numeric(df[tcol], errors="coerce")
        # drop improbable times (e.g., > 2359 or < 0)
        df.loc[(df[tcol] < 0) | (df[tcol] > 2359), tcol] = np.nan

# Drop rows missing schedule info (booking-time requirement)
need = [c for c in ["crs_dep_time","crs_arr_time"] if c in df.columns]
df = df.dropna(subset=need).copy()

# Build/normalize the label
LATE_THRESHOLD_MIN = 15
if "is_late" in df.columns:
    df["is_late"] = df["is_late"].astype("Int8")
elif "arr_delay" in df.columns:
    df["is_late"] = (df["arr_delay"] >= LATE_THRESHOLD_MIN).astype("Int8")
else:
    raise ValueError("Label not found after loading.")

In [ ]:
# Feature Engineering booking time only

def bucket_dep_hour(hs: pd.Series) -> pd.Series:
    """
    Bucket scheduled departure times into 4 bins:
    1: Early Morning (5AM-10:59AM)
    2: Late Morning to Afternoon (11AM-3:59PM)
    3: Late Afternoon to Evening (4PM-8:59PM)
    4: Night (9PM-4:59AM)
    0: Missing/Invalid
    Args:
        hs (pd.Series): Series of scheduled departure times in HHMM format.
    Returns:
        pd.Series: Series of integer bins (0-4) corresponding to departure time buckets.

    """
    h = (hs // 100).astype("int16")  # HHMM → HH
    out = np.zeros(len(h), dtype="int8")
    out[(h >= 5)  & (h < 11)] = 1
    out[(h >= 11) & (h < 16)] = 2
    out[(h >= 16) & (h < 21)] = 3
    out[(h >= 21)] = 4
    return pd.Series(out, index=h.index, dtype="int8")

df["dep_hour"] = (df["crs_dep_time"] // 100).astype("int8") # Extract departure hour
df["arr_hour"] = (df["crs_arr_time"] // 100).astype("int8") # Extract arrival hour

if "day_of_week" in df.columns:
    df["is_weekend"] = df["day_of_week"].isin([6,7]).astype("int8") # Weekend indicator
else:
    df["is_weekend"] = pd.Series(0, index=df.index, dtype="int8") # Default to 0 if day_of_week not present

df["dep_hour_bin"] = bucket_dep_hour(df["crs_dep_time"]) # Bucketed departure hour

if {"op_unique_carrier","origin"}.issubset(df.columns):
    df["carrier_origin_pair"] = (df["op_unique_carrier"].astype(str)
                                 + "_" + df["origin"].astype(str)).astype("category") # Carrier-Origin pair
else:
    df["carrier_origin_pair"] = "NA"
    df["carrier_origin_pair"] = df["carrier_origin_pair"].astype("category") # Default if columns not present


In [ ]:
# Keep only the columns on which our model needs to be train in future

CATEGORICAL_COLS = [
    "op_unique_carrier",
    "op_carrier_fl_num",
    "origin",
    "origin_city_name",
    "origin_state_nm",
    "dest",
    "dest_city_name",
    "dest_state_nm",
    "carrier_origin_pair"
]

NUMERIC_COLS = [
    "day_of_month",
    "day_of_week",
    "fl_date",
    "op_carrier_fl_num",
    "crs_dep_time",
    "dep_hour",
    "crs_arr_time",
    "arr_hour",
    "crs_elapsed_time",
    "distance",
    "is_weekend",
    "dep_hour_bin",
    "arr_delay"
]

keep = [c for c in CATEGORICAL_COLS + NUMERIC_COLS + ["is_late"] if c in df.columns] # Columns to keep
missing = sorted(set(CATEGORICAL_COLS + NUMERIC_COLS + ["is_late"]) - set(keep)) # Identify missing columns
if missing:
    print("WARNING - missing columns (they will be absent in output):", missing) 

df_clean = df.loc[:, keep].copy() # Select only the required columns
df_clean = df_clean.loc[:, ~df_clean.columns.duplicated()] # Remove duplicate columns
df_clean.reset_index(drop=True, inplace=True) # Reset index

print(df_clean.shape) # Print the shape of the cleaned dataframe
df_clean.head() # Display the first few rows of the cleaned dataframe

(7079080, 22)


,op_unique_carrier,op_carrier_fl_num,origin,origin_city_name,origin_state_nm,dest,dest_city_name,dest_state_nm,carrier_origin_pair,day_of_month,...,crs_dep_time,dep_hour,crs_arr_time,arr_hour,crs_elapsed_time,distance,is_weekend,dep_hour_bin,arr_delay,is_late
0,9E,4814.0,JFK,"New York, NY",New York,DTW,"Detroit, MI",Michigan,9E_JFK,1,...,1252.0,12,1508.0,15,136.0,509.0,0,2,-19.0,0
1,9E,4815.0,MSP,"Minneapolis, MN",Minnesota,CLE,"Cleveland, OH",Ohio,9E_MSP,1,...,1015.0,10,1325.0,13,130.0,622.0,0,1,-30.0,0
2,9E,4817.0,JFK,"New York, NY",New York,RIC,"Richmond, VA",Virginia,9E_JFK,1,...,1415.0,14,1601.0,16,106.0,288.0,0,2,-20.0,0
3,9E,4817.0,RIC,"Richmond, VA",Virginia,JFK,"New York, NY",New York,9E_RIC,1,...,1650.0,16,1841.0,18,111.0,288.0,0,3,-42.0,0
4,9E,4818.0,DTW,"Detroit, MI",Michigan,MKE,"Milwaukee, WI",Wisconsin,9E_DTW,1,...,1015.0,10,1034.0,10,79.0,237.0,0,1,-14.0,0


In [ ]:
# Column Profiling

def profile_dataframe(df: pd.DataFrame, top_k=5) -> pd.DataFrame:
    """
    Generate a profile summary for each column in the DataFrame.
    Args:
        df (pd.DataFrame): Input DataFrame to profile.
        top_k (int): Number of top frequent values to include for categorical columns.
    Returns:
        pd.DataFrame: DataFrame containing the profile summary for each column.
    """
    rows = []
    n = len(df)
    for c in df.columns:
        s = df[c]
        missing = int(s.isna().sum())
        dtype = str(s.dtype)

        if pd.api.types.is_numeric_dtype(s):
            desc = s.describe(percentiles=[.5,.95,.99])  # count, mean, std, min, 50%, 95%, 99%, max
            row = dict(
                column=c, dtype=dtype, non_null=int(desc["count"]),
                missing=missing, missing_pct=round(100*missing/n,2),
                mean=float(desc.get("mean", np.nan)),
                std=float(desc.get("std", np.nan)),
                min=float(desc.get("min", np.nan)),
                p50=float(desc.get("50%", np.nan)),
                p95=float(desc.get("95%", np.nan)),
                p99=float(desc.get("99%", np.nan)),
                max=float(desc.get("max", np.nan)),
                unique=np.nan, top_values=""
            ) # Numeric column profile
        else:
            vc = s.astype("string").value_counts(dropna=True).head(top_k)
            tops = "; ".join([f"{k}:{v}" for k, v in vc.items()])
            row = dict(
                column=c, dtype=dtype, non_null=n-missing,
                missing=missing, missing_pct=round(100*missing/n,2),
                mean=np.nan, std=np.nan, min=np.nan, p50=np.nan, p95=np.nan, p99=np.nan, max=np.nan,
                unique=int(s.nunique(dropna=True)), top_values=tops
            ) # Categorical column profile
        rows.append(row)
    return pd.DataFrame(rows)

profile_df = profile_dataframe(df_clean) # Generate the profile dataframe
profile_df.sort_values("column").head(12) # Display the first few rows of the profile dataframe


,column,dtype,non_null,missing,missing_pct,mean,std,min,p50,p95,p99,max,unique,top_values
20,arr_delay,float64,6965266,113814,1.61,7.098249,57.991274,-126.0,-6.0,83.0,215.0,3803.0,NaN,
15,arr_hour,int8,7079080,0,0.00,14.616533,5.187776,0.0,15.0,22.0,23.0,23.0,NaN,
8,carrier_origin_pair,category,7079080,0,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1738.0,DL_ATL:227628; AA_DFW:166286; AA_CLT:126484; U...
14,crs_arr_time,float64,7079080,0,0.00,1491.364958,519.438438,1.0,1517.0,2257.0,2355.0,2359.0,NaN,
12,crs_dep_time,float64,7079080,0,0.00,1327.299673,493.030491,1.0,1320.0,2124.0,2255.0,2359.0,NaN,
16,crs_elapsed_time,float64,7079079,1,0.00,146.766445,72.386903,-160.0,130.0,305.0,375.0,1326.0,NaN,
9,day_of_month,int64,7079080,0,0.00,15.784452,8.786433,1.0,16.0,29.0,31.0,31.0,NaN,
10,day_of_week,int64,7079080,0,0.00,3.981945,2.012279,1.0,4.0,7.0,7.0,7.0,NaN,
13,dep_hour,int8,7079080,0,0.00,13.003834,4.900544,0.0,13.0,21.0,22.0,23.0,NaN,
19,dep_hour_bin,int8,7079080,0,0.00,2.052009,0.950817,0.0,2.0,4.0,4.0,4.0,NaN,


In [ ]:
eda = {} # Quick EDA

# Late rate
eda["late_rate_overall"] = float(df_clean["is_late"].mean()) if "is_late" in df_clean else None # Overall late rate

# Late rate by airline (top 12 by volume)
if {"op_unique_carrier","is_late"}.issubset(df_clean.columns):
    grp = df_clean.groupby("op_unique_carrier")["is_late"].agg(["mean","count"]).sort_values("count", ascending=False).head(12) # Group by airline
    eda["late_rate_by_airline_top12"] = grp["mean"].round(3).to_dict() # Late rate by airline (top 12)

# Late rate by departure hour
if {"dep_hour","is_late"}.issubset(df_clean.columns):
    eda["late_rate_by_dep_hour"] = df_clean.groupby("dep_hour")["is_late"].mean().round(3).to_dict() # Late rate by departure hour

print(json.dumps(eda, indent=2)) # Print the EDA results in JSON format

/tmp/ipykernel_9692/4201265950.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grp = df_clean.groupby("op_unique_carrier")["is_late"].agg(["mean","count"]).sort_values("count", ascending=False).head(12)


{
  "late_rate_overall": 0.20482492075241415,
  "late_rate_by_airline_top12": {
    "WN": 0.204,
    "DL": 0.171,
    "AA": 0.257,
    "UA": 0.195,
    "OO": 0.187,
    "YX": 0.138,
    "MQ": 0.205,
    "NK": 0.234,
    "AS": 0.216,
    "B6": 0.25,
    "OH": 0.214,
    "F9": 0.28
  },
  "late_rate_by_dep_hour": {
    "0": 0.196,
    "1": 0.209,
    "2": 0.203,
    "3": 0.19,
    "4": 0.199,
    "5": 0.089,
    "6": 0.094,
    "7": 0.121,
    "8": 0.137,
    "9": 0.151,
    "10": 0.163,
    "11": 0.175,
    "12": 0.189,
    "13": 0.21,
    "14": 0.23,
    "15": 0.246,
    "16": 0.265,
    "17": 0.273,
    "18": 0.288,
    "19": 0.294,
    "20": 0.298,
    "21": 0.278,
    "22": 0.266,
    "23": 0.209
  }
}


In [ ]:
# Write cleaned CSV
tmp_clean = "/tmp/flight_data_2024_clean.csv"
df_clean.to_csv(tmp_clean, index=False)
s3.upload_file(tmp_clean, bucket_name, clean_key) # Upload cleaned data to S3

# Write profile CSV
tmp_prof = "/tmp/column_profile.csv"
profile_df.to_csv(tmp_prof, index=False)
s3.upload_file(tmp_prof, bucket_name, profile_key) # Upload profile data to S3

# Write EDA JSON
tmp_eda = "/tmp/quick_eda.json"
with open(tmp_eda, "w") as f:
    json.dump(eda, f, indent=2)
s3.upload_file(tmp_eda, bucket_name, eda_key) # Upload EDA JSON to S3

print("WROTE:")
print("  s3://%s/%s" % (bucket_name, clean_key)) # Print the S3 path of the cleaned data
print("  s3://%s/%s" % (bucket_name, profile_key)) # Print the S3 path of the profile data
print("  s3://%s/%s" % (bucket_name, eda_key)) # Print the S3 path of the EDA JSON

WROTE:
  s3://mlc-project-flight-data-2024/preprocessed/flight_data_2024_clean.csv
  s3://mlc-project-flight-data-2024/preprocessed/profiles/column_profile.csv
  s3://mlc-project-flight-data-2024/preprocessed/profiles/quick_eda.json
